# 1 · make — Top7 contact-map data

Folds one protein with one checkpoint and stores the raw prediction, so
[`1_plot_top7_heatmap.ipynb`](1_plot_top7_heatmap.ipynb) can draw it without a GPU.

Default protein is **Top7** (`1qys_A`), the de novo design from Kuhlman et al. 2003. It lives in
the legacy 554-protein universe, not in the FoldBench monomer sets.

The recipe is [#82](https://github.com/Open-Athena/MarinFold/issues/82)'s settled
**rollout + resample**: `N_ROLLOUTS` different realizations of the same protein's document
(contacts-v1 randomizes the N-terminal offset and the statement order), one sampled contact
section from each, votes counted per residue pair, ties broken by the pairwise log-probability.

**Needs a GPU.** Everything else in this directory runs on CPU.

In [ ]:
# Run from anywhere: figlib lives next to this notebook.
import sys
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "figlib.py").exists() else Path("experiments/exp250_evals_exploration_notebook/figures")
sys.path.insert(0, str(HERE.resolve()))
import figlib

In [ ]:
# --- parameters -------------------------------------------------------------------------------
DATASET = "1_top7_heatmap"
PROTEIN = "denovo_pdb__1qys_A"   # dataset__stem in the legacy 554
MODEL = "contacts-v1-exp232-m2-p06-train-1.5B"   # #232's best decontaminated final

N_ROLLOUTS = 100                 # exp82's settled count
TEMPERATURE = 1.0
TOP_P = 0.95
TOP_K = -1                       # disabled, as in the published harness
BACKEND = "auto"                 # auto -> vllm at compute capability >= 8.0, else transformers
DTYPE = "auto"                   # auto -> bfloat16; float16 overflows these weights (see README)
GPU_MEMORY_UTILIZATION = 0.85    # vLLM only; lower it if several engines share one card

PARAMETERS = dict(protein=PROTEIN, model=MODEL, n_rollouts=N_ROLLOUTS, temperature=TEMPERATURE,
                  top_p=TOP_P, top_k=TOP_K, backend=BACKEND, dtype=DTYPE,
                  gpu_memory_utilization=GPU_MEMORY_UTILIZATION)
PARAMETERS

In [ ]:
import importlib.util
import json

import numpy as np
import torch

from marinfold.document_structures.contacts_v1 import (
    GenerationConfig, InferenceConfig, RawContact, build_document, predict,
    residues_from_sequence, structure_from_sequence)


def resolve_backend(name: str) -> str:
    """vLLM where it is installed and supported (Ampere+), transformers otherwise."""
    if name == "transformers":
        return "transformers"
    usable = (importlib.util.find_spec("vllm") is not None and torch.cuda.is_available()
              and torch.cuda.get_device_capability()[0] >= 8)
    if name == "vllm" and not usable:
        print("note: vLLM asked for but not usable here — using transformers")
    return "vllm" if usable else "transformers"


def resolve_dtype(name: str) -> str:
    """bfloat16 on any GPU. float16 overflows this model's residual stream and dies in sampling."""
    if name != "auto":
        return name
    return "bfloat16" if torch.cuda.is_available() else "float32"


inputs = figlib.Inputs()
targets, ground_truth = figlib.load_legacy_universe(inputs)
dataset_name, stem = PROTEIN.split("__", 1)
target = targets[(targets.dataset == dataset_name) & (targets.stem == stem)].iloc[0]
truth = ground_truth[(dataset_name, stem)]
print(f"{PROTEIN}  L={target.L}  {len(truth['contacts'])} ground-truth contacts  "
      f"{len(truth['resolved'])} resolved residues")

In [ ]:
backend, dtype = resolve_backend(BACKEND), resolve_dtype(DTYPE)
model_identity = figlib.model_identity(MODEL)
print(f"backend {backend} · dtype {dtype} · rope_theta {model_identity['rope_theta']}")

config = InferenceConfig(
    model=MODEL, backend=backend, method="rollout", keep_matrix=True,
    n_rollouts=N_ROLLOUTS, temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
    dtype=dtype, min_seq_separation=figlib.MIN_SEPARATION,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION)

record = next(iter(predict(config, structures=[
    structure_from_sequence(target.input_seq, entry_id=stem)])))
# The band within MIN_SEPARATION comes back as NaN — never a candidate pair, but NaN poisons
# argsort, so it is pushed below every real score.
score = np.nan_to_num(np.asarray(record["score_matrix"], dtype=float), nan=-1e9)
# `predict` returns votes + a tie-break term in [0, 0.5); the integer part is the vote count.
votes = np.where(score > -1e8, np.floor(score), 0).astype(int)
assert votes.max() <= N_ROLLOUTS, f"{votes.max()} votes from {N_ROLLOUTS} rollouts"
metrics = figlib.score_metrics(score, truth)
headline = metrics[(metrics.range == "all") & (metrics.cut == "R")].iloc[0]
print(f"R-precision {headline.value:.3f} over {int(headline.n_true)} true contacts "
      f"({int(headline.n_candidate)} candidate pairs)")
metrics[metrics.cut.isin(["R", "AUC"])]

In [ ]:
# The deposited structure's CA trace, so the plot notebook can draw the fold beside the map.
# The chain is the one #89 scored (`gt_chain`), and the CA count must equal the resolved-residue
# count — if it does not, the trace and the contact map would be indexed differently and the
# ribbon would line up with nothing.
import gemmi

pdb_id = stem.split("_")[0].upper()
structure_url = f"https://files.rcsb.org/download/{pdb_id}-assembly1.cif"
raw = inputs.fetch(structure_url).decode()
block = gemmi.cif.read_string(raw).sole_block()
deposited = gemmi.make_structure_from_block(block)
deposited.setup_entities()
chain = next(ch for ch in deposited[0] if ch.name == truth["gt_chain"])
alpha_carbons = [residue.find_atom("CA", "*") for residue in chain]
alpha_carbons = [atom for atom in alpha_carbons if atom is not None]
resolved = np.asarray(truth["resolved"])
if len(alpha_carbons) != len(resolved):
    raise SystemExit(f"{pdb_id} chain {truth['gt_chain']} has {len(alpha_carbons)} CA atoms but "
                     f"#89 resolved {len(resolved)} residues — the mapping from structure to "
                     f"contact-map index is not 1:1 and the ribbon would be misaligned")
ca_coords = np.full((int(truth["L"]), 3), np.nan)
ca_coords[resolved] = [[atom.pos.x, atom.pos.y, atom.pos.z] for atom in alpha_carbons]
# The author residue numbers, so a renderer can colour `resi N` in the deposited numbering while
# the contact map is indexed from 0. 1QYS chain A runs 3..94 against sequence indices 0..91, so
# the two are not interchangeable.
residue_numbers = np.full(int(truth["L"]), -1, dtype=int)
residue_numbers[resolved] = [residue.seqid.num for residue in chain
                             if residue.find_atom("CA", "*") is not None]
print(f"{pdb_id} chain {truth['gt_chain']}: {len(alpha_carbons)} CA atoms mapped onto "
      f"{len(resolved)} resolved residues")

# The document itself, so a format panel can be drawn from the real thing rather than a paraphrase.
document = build_document(
    stem, residues_from_sequence(target.input_seq),
    [RawContact(seq_i=int(i), seq_j=int(j), degree=float(d)) for i, j, d in truth["contacts"]],
    config=GenerationConfig())

figlib.write_dataset(
    DATASET,
    notebook="1_make_top7_heatmap_data.ipynb",
    parameters=PARAMETERS,
    inputs=inputs,
    files={
        # Two matrices, because they answer different questions and only one is interpretable.
        # `score` is what the metric ranks: votes plus the pairwise tie-break, which adds a
        # fraction in [0, 0.5) purely to order pairs that are tied on votes (mostly the large
        # zero-vote mass). `votes` is the count itself — how many of the N_ROLLOUTS rollouts
        # asserted that pair — recovered as the integer part, and the only one of the two that
        # means anything on its own.
        "score.npy": lambda path: np.save(path, score.astype(np.float32)),
        "votes.npy": lambda path: np.save(path, votes.astype(np.int16)),
        "ground_truth.json": json.dumps({
            "dataset": dataset_name, "stem": stem, "L": int(truth["L"]),
            "resolved": [int(i) for i in truth["resolved"]],
            "contacts": [[int(i), int(j), float(d)] for i, j, d in truth["contacts"]],
            "sequence": target.input_seq,
        }, indent=2).encode(),
        "metrics.csv": lambda path: metrics.to_csv(path, index=False),
        "ca_coords.npy": lambda path: np.save(path, ca_coords.astype(np.float32)),
        # The deposited coordinates themselves, so the cartoon renderers in the plot notebook do
        # not have to re-fetch (and cannot quietly get a different revision than the CA trace).
        "structure.cif": raw.encode(),
        "residue_numbers.npy": lambda path: np.save(path, residue_numbers),
        "document.txt": document.document.encode(),
    },
    extra={
        "model": model_identity,
        "recipe": {"method": "rollout+resample+tiebreak", "backend": backend, "dtype": dtype,
                   "n_rollouts": N_ROLLOUTS, "temperature": TEMPERATURE, "top_p": TOP_P,
                   "top_k": TOP_K, "min_seq_separation": figlib.MIN_SEPARATION},
        "protein": {"dataset": dataset_name, "stem": stem, "L": int(target.L),
                    "sequence_sha256": figlib.digest(target.input_seq.encode()),
                    "n_true_contacts": int(headline.n_true),
                    "n_candidate_pairs": int(headline.n_candidate)},
        "structure": {"pdb_id": pdb_id, "chain": truth["gt_chain"], "source": structure_url,
                      "n_ca": len(alpha_carbons),
                      "note": "CA coordinates in input-sequence index order; NaN where the "
                              "residue is unresolved. Same indexing as the contact map."},
        "votes": {"max": int(votes.max()), "n_rollouts": N_ROLLOUTS,
                  "pairs_with_any_vote": int((np.triu(votes, k=1) > 0).sum()),
                  "note": "score.npy = votes.npy + a pairwise tie-break in [0, 0.5); the metric "
                          "ranks score, the figure shows votes / n_rollouts"},
        "result": {"r_precision_all": float(headline.value),
                   "auc_all": float(metrics[(metrics.range == "all")
                                            & (metrics.cut == "AUC")].value.iloc[0])},
    })